In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder \
    .appName("RFM_Pipeline") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

26/04/16 12:57:48 WARN Utils: Your hostname, Jonathans-MacBook-Pro-16.local resolves to a loopback address: 127.0.0.1; using 10.55.66.254 instead (on interface en0)
26/04/16 12:57:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 12:57:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/16 12:57:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
pdf = pd.read_excel("../data/raw/online_retail_II.xlsx", engine="openpyxl")
df = spark.createDataFrame(pdf)

In [4]:
df = df.withColumn("TotalPrice", col("Quantity") * col("Price"))
df.show(5)

26/04/16 13:03:55 WARN TaskSetManager: Stage 0 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|        TotalPrice|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|              83.4|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|              81.0|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|              81.0|
| 489434|    22041|RECORD FRAME 7" S...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|100.80000000000001|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|              30.0|
+-------+---------+-----

In [ ]:
# Data Cleaning
df_clean = (df
    .dropDuplicates()                                           # Only entire row duplicates
    .filter(col("Quantity") > 0 & col("UnitPrice") > 0)         # Valid transactions only
    .withColumn("Customer ID", col("Customer ID").cast("int"))
    .withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))
)

print(f"Cleaned: {df_clean.count()} rows from {df.count()} original")
print("Data quality: No missing values, only row-level duplicates removed")
df_clean.show(5)